# Modul Pembelajaran Constraint Satisfaction Problems (CSP) RKA 25

## 1. Pendahuluan
**Constraint Satisfaction Problems (CSP)** adalah kerangka kerja untuk memecahkan masalah di mana kita harus **menentukan nilai untuk sekumpulan variabel** sedemikian rupa sehingga semua **batasan (constraints) terpenuhi**.

CSP bukan mencari **jalur** menuju tujuan, melainkan mencari **penugasan (assignment)** yang valid.

> *Pertanyaan CSP: Nilai apa yang harus diberikan ke setiap variabel agar semua aturan terpenuhi?*

Dua algoritma utama yang akan dipelajari:
1. **Backtracking Search** — algoritma dasar untuk CSP
2. **Forward Checking** — optimasi Backtracking dengan pruning domain lebih awal

## 2. Konsep Dasar



### 2.1 Komponen Formal CSP
![image.png](attachment:image.png)

Sebuah CSP didefinisikan oleh tiga komponen:

| Komponen | Deskripsi | Contoh (Map Coloring) |
|---|---|---|
| **Variables (X)** | Sekumpulan variabel yang harus diberi nilai | WA, NT, Q, NSW, V, SA, T |
| **Domains (D)** | Himpunan nilai yang boleh diambil tiap variabel | {merah, hijau, biru} |
| **Constraints (C)** | Aturan yang membatasi kombinasi nilai antar variabel | WA ≠ NT, WA ≠ SA, NT ≠ SA, ... |

**Solusi CSP** = assignment lengkap yang memenuhi semua constraint.

Perbedaan mendasar CSP dengan search biasa:
- Search biasa: state adalah *black box*, goal test bisa apa saja
- CSP: state terdefinisi oleh variabel dan domain, goal test berupa kumpulan constraint — struktur ini memungkinkan algoritma yang jauh lebih efisien

### 2.2 Jenis Constraint

Constraint dalam CSP dibedakan berdasarkan jumlah variabel yang terlibat:

- **Unary constraint**: melibatkan satu variabel. Ekuivalen dengan memperkecil domain.
  
  Contoh: `SA ≠ hijau`

- **Binary constraint**: melibatkan dua variabel. Paling umum digunakan.
  
  Contoh: `SA ≠ WA`

- **Higher-order constraint**: melibatkan tiga variabel atau lebih.
  
  Contoh: constraint pada kriptaritmetik (SEND + MORE = MONEY)

### 2.3 Contoh Klasik: Map Coloring Australia

Masalah: warnai peta Australia sedemikian rupa sehingga tidak ada dua wilayah yang bertetangga memiliki warna yang sama.
![image.png](attachment:image.png)

```
Variabel : WA, NT, Q, NSW, V, SA, T
Domain   : {merah, hijau, biru}
Constraint (binary): WA≠NT, WA≠SA, NT≠SA, NT≠Q, SA≠Q, SA≠NSW, SA≠V, Q≠NSW, NSW≠V
```

Salah satu solusi yang valid:
```
WA=merah, NT=hijau, Q=merah, NSW=hijau, V=merah, SA=biru, T=merah
```

**Constraint Graph** — cara visual untuk memetakan constraint binary:
- Node = variabel
- Edge = constraint antara dua variabel

```
  WA --- NT --- Q
   \   /   \  /
    \ /     SA
     SA --- NSW
       \   /
         V

       T  (isolated — tidak ada tetangga)
```

Perhatikan: **T (Tasmania) terisolasi** — tidak punya constraint dengan wilayah lain, sehingga bisa diselesaikan terpisah sebagai subproblem independen.

## 3. Backtracking Search

### 3.1 Penjelasan Backtracking

**Backtracking Search** adalah algoritma DFS yang disesuaikan untuk CSP, dengan dua perbaikan penting dibanding DFS naif:

1. **One variable at a time** - setiap langkah hanya menugaskan nilai ke satu variabel. Urutan variabel bersifat *commutative*, sehingga kita cukup pilih satu urutan tetap.

2. **Check constraints as you go** - sebelum menugaskan nilai, cek apakah nilai tersebut konsisten dengan assignment sebelumnya. Jika tidak, skip nilai itu (*incremental goal test*).

Jika seluruh nilai suatu variabel sudah dicoba dan tidak ada yang konsisten, **backtrack** ke variabel sebelumnya dan coba nilai lain.

Pseudocode:
```
function BACKTRACKING(assignment, csp):
    if assignment lengkap: return assignment
    var = pilih variabel yang belum diassign
    for value in domain[var]:
        if value konsisten dengan assignment:
            tambahkan {var=value} ke assignment
            result = BACKTRACKING(assignment, csp)
            if result != failure: return result
            hapus {var=value} dari assignment
    return failure
```

**Properti Backtracking:**
- **Complete**: Ya — pasti menemukan solusi jika ada
- **Optimal**: Tidak relevan — semua solusi valid setara
- **Time Complexity**: O(d^n) — d = ukuran domain, n = jumlah variabel

### 3.2 Representasi CSP

Kita representasikan CSP Map Coloring Australia sebagai:
- `variables`: list variabel
- `domains`: dictionary variabel -> list nilai yang mungkin
- `constraints`: list tuple pasangan variabel yang tidak boleh sama

In [1]:
variables = ['WA', 'NT', 'Q', 'NSW', 'V', 'SA', 'T']

domains = {
    'WA':  ['merah', 'hijau', 'biru'],
    'NT':  ['merah', 'hijau', 'biru'],
    'Q':   ['merah', 'hijau', 'biru'],
    'NSW': ['merah', 'hijau', 'biru'],
    'V':   ['merah', 'hijau', 'biru'],
    'SA':  ['merah', 'hijau', 'biru'],
    'T':   ['merah', 'hijau', 'biru'],
}

# Pasangan wilayah yang bertetangga -> tidak boleh warna sama
constraints = [
    ('WA', 'NT'), ('WA', 'SA'),
    ('NT', 'SA'), ('NT', 'Q'),
    ('SA', 'Q'),  ('SA', 'NSW'), ('SA', 'V'),
    ('Q',  'NSW'),
    ('NSW','V'),
]

### 3.3 Fungsi Pengecekan Konsistensi

Fungsi `is_consistent` mengecek apakah penugasan `var=value` tidak melanggar constraint manapun dengan variabel yang sudah diassign sebelumnya.

In [2]:
def is_consistent(var, value, assignment, constraints):
    for (v1, v2) in constraints:
        # Cek apakah var terlibat dalam constraint ini
        if v1 == var and v2 in assignment:
            if assignment[v2] == value:
                return False
        if v2 == var and v1 in assignment:
            if assignment[v1] == value:
                return False
    return True

### 3.4 Implementasi Backtracking

Implementasi rekursif: pilih variabel yang belum diassign, coba setiap nilai di domain-nya, lanjutkan rekursi jika konsisten, backtrack jika gagal.

In [3]:
def backtracking(assignment, variables, domains, constraints):

    # Base case: semua variabel sudah diassign
    if len(assignment) == len(variables):
        return assignment

    # Pilih variabel berikutnya yang belum diassign
    unassigned = [v for v in variables if v not in assignment]
    var = unassigned[0]

    for value in domains[var]:
        if is_consistent(var, value, assignment, constraints):
            assignment[var] = value

            result = backtracking(assignment, variables, domains, constraints)
            if result is not None:
                return result

            # Backtrack: hapus assignment yang tidak berhasil
            del assignment[var]

    return None

### 3.5 Menjalankan Backtracking

Mulai dari assignment kosong dan jalankan algoritma.

In [4]:
solution = backtracking({}, variables, domains, constraints)

if solution:
    for var, color in solution.items():
        print(f"{var} = {color}")
else:
    print("Tidak ada solusi.")

WA = merah
NT = hijau
Q = merah
NSW = hijau
V = merah
SA = biru
T = merah


## 4. Peningkatan Backtracking: Forward Checking

### 4.1 Motivasi Forward Checking

Masalah Backtracking dasar: kegagalan baru terdeteksi **setelah** variabel diassign. Kita baru tahu sebuah nilai buruk setelah mencoba semua cabang di bawahnya.

**Forward Checking** mendeteksi kegagalan **lebih awal** dengan cara:

> Setiap kali variabel X diassign nilai v, hapus dari domain semua variabel tetangga Y nilai-nilai yang **bertentangan** dengan `X=v`.

Jika domain suatu variabel menjadi **kosong**, langsung backtrack — tidak perlu melanjutkan eksplorasi cabang ini.

Ilustrasi pada Map Coloring:
```
WA = merah  →  hapus 'merah' dari domain NT dan SA
NT = hijau  →  hapus 'hijau' dari domain SA dan Q
Q  = merah  →  hapus 'merah' dari domain NSW dan SA
                SA sekarang domain-nya = {biru} ← hanya satu pilihan!
```

Forward Checking adalah bentuk **constraint propagation** — menyebarkan dampak suatu assignment ke variabel-variabel yang belum diassign.

### 4.2 Implementasi Forward Checking

Kita perlu menyimpan **salinan domain** yang bisa dimodifikasi selama pencarian, agar saat backtrack domain bisa dikembalikan.

In [5]:
import copy

def get_neighbors(var, constraints):
    """Mengembalikan semua variabel yang memiliki constraint dengan var."""
    neighbors = []
    for (v1, v2) in constraints:
        if v1 == var:
            neighbors.append(v2)
        elif v2 == var:
            neighbors.append(v1)
    return neighbors

In [6]:
def forward_checking(assignment, variables, domains, constraints):

    # Base case: semua variabel sudah diassign
    if len(assignment) == len(variables):
        return assignment

    unassigned = [v for v in variables if v not in assignment]
    var = unassigned[0]

    for value in domains[var]:
        if is_consistent(var, value, assignment, constraints):
            assignment[var] = value

            # Simpan domain saat ini sebelum dimodifikasi
            saved_domains = copy.deepcopy(domains)

            # Propagasi: hapus nilai yang bertentangan dari domain tetangga
            domain_wiped = False
            for neighbor in get_neighbors(var, constraints):
                if neighbor not in assignment:
                    domains[neighbor] = [
                        v for v in domains[neighbor] if v != value
                    ]
                    # Jika domain tetangga kosong, hentikan propagasi
                    if len(domains[neighbor]) == 0:
                        domain_wiped = True
                        break

            if not domain_wiped:
                result = forward_checking(assignment, variables, domains, constraints)
                if result is not None:
                    return result

            # Backtrack: kembalikan domain dan hapus assignment
            domains.update(saved_domains)
            del assignment[var]

    return None

### 4.3 Menjalankan Forward Checking

In [7]:
# Reset domain ke kondisi awal sebelum dijalankan
domains_fc = {
    'WA':  ['merah', 'hijau', 'biru'],
    'NT':  ['merah', 'hijau', 'biru'],
    'Q':   ['merah', 'hijau', 'biru'],
    'NSW': ['merah', 'hijau', 'biru'],
    'V':   ['merah', 'hijau', 'biru'],
    'SA':  ['merah', 'hijau', 'biru'],
    'T':   ['merah', 'hijau', 'biru'],
}

solution_fc = forward_checking({}, variables, domains_fc, constraints)

if solution_fc:
    for var, color in solution_fc.items():
        print(f"{var} = {color}")
else:
    print("Tidak ada solusi.")

WA = merah
NT = hijau
Q = merah
NSW = hijau
V = merah
SA = biru
T = merah


## 5. Heuristik Ordering

### 5.1 Minimum Remaining Values (MRV)

Pada Backtracking dasar, urutan pemilihan variabel bersifat statis (sesuai urutan list). Heuristik **MRV** mengubah ini:

> Selalu pilih variabel yang domain-nya **paling sedikit** tersisa.

Intuisi: variabel dengan domain terkecil paling mungkin gagal lebih awal (*fail-fast*). Dengan memprioritaskannya, kita bisa backtrack lebih cepat tanpa membuang waktu mengeksplorasi cabang yang pasti gagal.

MRV disebut juga **most constrained variable** — pilih variabel yang paling banyak terbebani constraint.

In [8]:
def select_mrv(variables, assignment, domains):
    """Pilih variabel unassigned dengan domain terkecil (MRV)."""
    unassigned = [v for v in variables if v not in assignment]
    return min(unassigned, key=lambda v: len(domains[v]))

### 5.2 Least Constraining Value (LCV)

Setelah memilih variabel dengan MRV, kita perlu memilih **nilai mana** yang dicoba pertama. Heuristik **LCV** menjawab ini:

> Pilih nilai yang **paling sedikit membatasi** pilihan variabel-variabel tetangga yang belum diassign.

Intuisi: kita ingin menjaga pilihan sebanyak mungkin untuk variabel lain agar pencarian tetap fleksibel. Nilai yang mengeliminasi paling sedikit pilihan tetangga lebih aman dicoba duluan.

In [9]:
def order_lcv(var, domains, assignment, constraints):
    """Urutkan nilai di domain var dari yang paling sedikit membatasi tetangga."""
    def count_conflicts(value):
        total = 0
        for neighbor in get_neighbors(var, constraints):
            if neighbor not in assignment:
                # Hitung berapa nilai di domain tetangga yang akan tereliminasi
                total += domains[neighbor].count(value)
        return total

    return sorted(domains[var], key=count_conflicts)

### 5.3 Backtracking dengan MRV + LCV

Gabungkan kedua heuristik ke dalam Backtracking dengan Forward Checking untuk mendapatkan implementasi yang lebih efisien.

In [10]:
def backtracking_mrv_lcv(assignment, variables, domains, constraints):

    if len(assignment) == len(variables):
        return assignment

    # Pilih variabel dengan domain terkecil (MRV)
    var = select_mrv(variables, assignment, domains)

    # Urutkan nilai dari yang paling sedikit membatasi (LCV)
    ordered_values = order_lcv(var, domains, assignment, constraints)

    for value in ordered_values:
        if is_consistent(var, value, assignment, constraints):
            assignment[var] = value
            saved_domains = copy.deepcopy(domains)

            domain_wiped = False
            for neighbor in get_neighbors(var, constraints):
                if neighbor not in assignment:
                    domains[neighbor] = [
                        v for v in domains[neighbor] if v != value
                    ]
                    if len(domains[neighbor]) == 0:
                        domain_wiped = True
                        break

            if not domain_wiped:
                result = backtracking_mrv_lcv(assignment, variables, domains, constraints)
                if result is not None:
                    return result

            domains.update(saved_domains)
            del assignment[var]

    return None

In [11]:
domains_mrv = {
    'WA':  ['merah', 'hijau', 'biru'],
    'NT':  ['merah', 'hijau', 'biru'],
    'Q':   ['merah', 'hijau', 'biru'],
    'NSW': ['merah', 'hijau', 'biru'],
    'V':   ['merah', 'hijau', 'biru'],
    'SA':  ['merah', 'hijau', 'biru'],
    'T':   ['merah', 'hijau', 'biru'],
}

solution_mrv = backtracking_mrv_lcv({}, variables, domains_mrv, constraints)

if solution_mrv:
    for var, color in solution_mrv.items():
        print(f"{var} = {color}")
else:
    print("Tidak ada solusi.")

WA = merah
NT = hijau
SA = biru
Q = merah
NSW = hijau
V = merah
T = merah


## 6. Perbandingan dan Ringkasan

### 6.1 Perbandingan Pendekatan

| Aspek | Backtracking | Forward Checking | FC + MRV + LCV |
|---|---|---|---|
| **Deteksi kegagalan** | Setelah assignment | Saat propagasi domain | Saat propagasi domain |
| **Pemilihan variabel** | Statis (urutan list) | Statis | Dinamis (MRV) |
| **Pemilihan nilai** | Urutan domain | Urutan domain | Cerdas (LCV) |
| **Overhead** | Rendah | Sedang | Sedang-tinggi |
| **Efisiensi** | Paling lambat | Lebih cepat | Paling cepat |

**Catatan:** Forward Checking dan MRV+LCV menghasilkan solusi yang **sama valid**, hanya berbeda pada **efisiensi pencarian** (berapa banyak node yang dieksplorasi sebelum solusi ditemukan).

### 6.2 Kapan Menggunakan CSP?

Gunakan pendekatan CSP ketika:
- Masalah melibatkan **penugasan nilai ke variabel** dengan aturan-aturan tertentu
- Tujuan adalah **menemukan assignment yang valid**, bukan jalur terpendek
- Terdapat **struktur constraint** yang bisa dieksploitasi untuk efisiensi

Contoh aplikasi nyata:
- **Penjadwalan**: penjadwalan ujian (ruang, waktu, mata kuliah, dosen)
- **Puzzle**: Sudoku, n-Queens, crossword puzzle
- **Konfigurasi**: konfigurasi hardware atau software dengan aturan kompatibilitas
- **Pewarnaan graf**: alokasi register di compiler, frequency assignment di jaringan seluler

## 7. Latihan Soal (PR)
<font color="blue">Latihan 5</font>

### PR — N-Queens dengan Backtracking & Forward Checking

**Tujuan:**
- Mengimplementasikan CSP untuk masalah N-Queens
- Memahami cara merepresentasikan masalah sebagai variabel, domain, dan constraint
- Mengimplementasikan Backtracking dan Forward Checking

**Deskripsi Masalah:**

Tempatkan N buah ratu pada papan catur N×N sedemikian rupa sehingga tidak ada dua ratu yang saling menyerang (tidak sebaris, sekolom, atau sediagonal).

**Formasi CSP:**
- Variabel: `Q1, Q2, ..., QN` (satu per baris)
- Domain: `{1, 2, ..., N}` (kolom tempat ratu diletakkan)
- Constraint: `non_attacking(Qi, Qj)` untuk semua `i ≠ j`

**Instruksi:** Lengkapi bagian kode yang kosong (`_____`).

### 1. Setup

In [12]:
N = 6  # Ukuran papan catur (bisa diubah)

variables_q = [f'Q{i}' for i in range(1, N+1)]

domains_q = {var: list(range(1, N+1)) for var in variables_q}

### 2. Fungsi Constraint

In [13]:
def queens_consistent(var, value, assignment):
    """
    Cek apakah menempatkan ratu di kolom `value` pada baris `var`
    tidak menyerang ratu yang sudah ditempatkan sebelumnya.

    Ratu di baris i kolom c1 menyerang ratu di baris j kolom c2 jika:
    - c1 == c2          (kolom sama)
    - |i-j| == |c1-c2|  (diagonal sama)
    """
    row1 = int(var[1:])  # nomor baris var
    c1 = value           # kolom ratu saat ini

    for other_var, other_value in assignment.items():
        row2 = int(other_var[1:])
        c2 = other_value # kolom ratu yang sudah diassign

        # Cek kolom sama
        if c1 == c2:
            return False

        # Cek diagonal
        if abs(row1 - row2) == abs(c1 - c2):
            return False

    return True

### 3. Backtracking untuk N-Queens

In [14]:
def backtracking_queens(assignment, variables, domains):

    if len(assignment) == len(variables):
        return assignment

    unassigned = [v for v in variables if v not in assignment]
    var = unassigned[0]

    for value in domains[var]:
        if queens_consistent(var, value, assignment):
            assignment[var] = value

            result = backtracking_queens(assignment, variables, domains)
            if result is not None:
                return result

            del assignment[var]

    return None

### 4. Forward Checking untuk N-Queens

In [15]:
import copy

def forward_checking_queens(assignment, variables, domains):

    if len(assignment) == len(variables):
        return assignment

    unassigned = [v for v in variables if v not in assignment]

    # Gunakan MRV: pilih variabel dengan domain terkecil
    var = min(unassigned, key=lambda v: len(domains[v]))

    for value in domains[var]:
        if queens_consistent(var, value, assignment):
            assignment[var] = value
            saved_domains = copy.deepcopy(domains)

            domain_wiped = False
            row1 = int(var[1:])

            for other_var in unassigned:
                if other_var == var or other_var in assignment:
                    continue

                row2 = int(other_var[1:])

                # Hapus nilai yang bertentangan dari domain tetangga
                domains[other_var] = [
                    v for v in domains[other_var]
                    if v != value and abs(row1 - row2) != abs(value - v)
                ]

                if len(domains[other_var]) == 0:
                    domain_wiped = True
                    break

            if not domain_wiped:
                result = forward_checking_queens(assignment, variables, domains)
                if result is not None:
                    return result

            domains.update(saved_domains)
            del assignment[var]

    return None

### 5. Fungsi Visualisasi Papan

In [16]:
def print_board(solution, n):
    for row in range(1, n+1):
        col = solution[f'Q{row}']
        line = ''
        for c in range(1, n+1):
            line += 'Q ' if c == col else '. '
        print(line)

### 6. Uji Coba

In [17]:
# Jalankan Backtracking
sol_bt = backtracking_queens({}, variables_q, copy.deepcopy(domains_q))

if sol_bt:
    print(f"Solusi Backtracking ({N}-Queens):")
    print_board(sol_bt, N)
else:
    print("Tidak ada solusi.")

Solusi Backtracking (6-Queens):
. Q . . . . 
. . . Q . . 
. . . . . Q 
Q . . . . . 
. . Q . . . 
. . . . Q . 


In [18]:
# Jalankan Forward Checking
sol_fc = forward_checking_queens({}, variables_q, copy.deepcopy(domains_q))

if sol_fc:
    print(f"Solusi Forward Checking ({N}-Queens):")
    print_board(sol_fc, N)
else:
    print("Tidak ada solusi.")

Solusi Forward Checking (6-Queens):
. Q . . . . 
. . . Q . . 
. . . . . Q 
Q . . . . . 
. . Q . . . 
. . . . Q . 
